# Cross-encoder reranker (Kaggle GPU)

Fine-tunes `cross-encoder/ms-marco-MiniLM-L-6-v2` (Apache-2.0, 22M params) on record pairs from the
LightGBM model's uncertain band, then scores every attached pair file (`ce_check.parquet`,
`ce_test.parquet`). **Settings: Accelerator = GPU (T4 or P100), Internet = On.**
Outputs in `/kaggle/working`: `ce_model/`, `ce_scores_<name>.parquet` (s1_eid, cand_eid, ce).

In [ ]:
import glob, os, time, math
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
BASE = "cross-encoder/ms-marco-MiniLM-L-6-v2"
OUT = "/kaggle/working"; MODEL_DIR = f"{OUT}/ce_model"
EPOCHS, BATCH, LR, MAXLEN = 2, 256, 3e-5, 128
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device", dev, torch.cuda.get_device_name(0) if dev == "cuda" else "")
files = {os.path.basename(p): p for p in glob.glob("/kaggle/input/**/*.parquet", recursive=True)}
print(sorted(files))

In [ ]:
# model: reuse a trained one if present (this session or an attached earlier output), else train
cands = [MODEL_DIR] + sorted(glob.glob("/kaggle/input/**/ce_model", recursive=True))
found = next((d for d in cands if os.path.exists(os.path.join(d, "config.json"))), None)
tok = AutoTokenizer.from_pretrained(found or BASE)
model = AutoModelForSequenceClassification.from_pretrained(found or BASE, num_labels=1).to(dev)

def batches(a, b, bs):
    for i in range(0, len(a), bs):
        enc = tok(list(a[i:i + bs]), list(b[i:i + bs]), padding=True, truncation=True, max_length=MAXLEN, return_tensors="pt")
        yield {k: v.to(dev, non_blocking=True) for k, v in enc.items()}

if found:
    print("loaded trained model from", found)
else:
    tr = pd.read_parquet(files["ce_train.parquet"], columns=["s1_eid", "text_a", "text_b", "y"])
    rng = np.random.default_rng(0)
    s1 = tr["s1_eid"].unique(); hold = set(rng.choice(s1, size=len(s1) // 50, replace=False))
    va = tr[tr["s1_eid"].isin(hold)]; tr = tr[~tr["s1_eid"].isin(hold)].sample(frac=1.0, random_state=0)
    print(f"train {len(tr):,} pairs, holdout {len(va):,}")
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * math.ceil(len(tr) / BATCH); warm = int(0.05 * steps)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / max(warm, 1)) * max(0.0, (steps - s) / max(steps - warm, 1)))
    scaler = torch.cuda.amp.GradScaler(enabled=dev == "cuda")
    lossf = torch.nn.BCEWithLogitsLoss()
    step, t0 = 0, time.time()
    for ep in range(EPOCHS):
        model.train()
        order = np.random.default_rng(ep).permutation(len(tr))
        a, b, y = tr["text_a"].values[order], tr["text_b"].values[order], tr["y"].values[order].astype(np.float32)
        for i, enc in enumerate(batches(a, b, BATCH)):
            yy = torch.tensor(y[i * BATCH:(i + 1) * BATCH], device=dev)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
                loss = lossf(model(**enc).logits.squeeze(-1), yy)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step(); step += 1
            if step % 500 == 0:
                print(f"epoch {ep} step {step}/{steps} loss {loss.item():.4f} ({time.time() - t0:.0f}s)", flush=True)
    model.save_pretrained(MODEL_DIR); tok.save_pretrained(MODEL_DIR)
    print("saved", MODEL_DIR, f"({time.time() - t0:.0f}s)")

In [ ]:
@torch.no_grad()
def score(df, bs=1024):
    model.eval()
    order = np.argsort(df["text_a"].str.len().values + df["text_b"].str.len().values)  # length-sorted: less padding
    a, b = df["text_a"].values[order], df["text_b"].values[order]
    out = np.empty(len(df), dtype=np.float32); o = 0
    for enc in batches(a, b, bs):
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
            z = model(**enc).logits.squeeze(-1).float()
        out[o:o + len(z)] = torch.sigmoid(z).cpu().numpy(); o += len(z)
    res = np.empty_like(out); res[order] = out
    return res

if "va" in globals() and len(va):
    from sklearn.metrics import roc_auc_score, average_precision_score
    pv = score(va); print(f"holdout AUC {roc_auc_score(va['y'], pv):.5f}  AP {average_precision_score(va['y'], pv):.5f}")
for name in ("ce_check.parquet", "ce_test.parquet"):
    if name in files:
        t0 = time.time(); df = pd.read_parquet(files[name])
        df["ce"] = score(df)
        dst = f"{OUT}/ce_scores_{name.split('_')[1]}"
        df[["s1_eid", "cand_eid", "ce"]].to_parquet(dst, index=False)
        print(f"{name}: {len(df):,} pairs scored -> {dst} ({time.time() - t0:.0f}s)")